# spark.read — NDJSON, one object per line (try it live)

Read **NDJSON** (JSON Lines), where the line *is* the record — and see what
escaping does to values full of quotes, tabs and newlines.

Companion to the note: **5. spark.read — NDJSON, one object per line**
on [ravi-writes.pages.dev](https://ravi-writes.pages.dev/playground/read-ndjson-escaped-values).

Run top to bottom: **Runtime → Run all**.

## The data & the requirement

**NDJSON** is one JSON object per **physical line** — no wrapping `[ ]`, no commas
between records:

```
{"id": 1, "customer": "Asha", "amount": 120.5}
{"id": 2, "customer": "Ben",  "amount": 0.0}
```

That is what `spark.read.format("json")` expects **by default**, and it is why
machines write logs and feeds this way: a reader can start anywhere, find the next
newline, and be on a record boundary.

Which raises the question this example is about: **what if a value itself contains a
newline?**

The cell below builds the file with **plain Python** — the input has to arrive from
*outside* Spark, or reading it with Spark proves nothing. Nine records go through
`json.dumps`, which does the escaping for you; the tenth is written **raw, by hand**.

In [ ]:
import json, os

INPUT_PATH = "/content/ravi-writes/data/input/orders.ndjson"
os.makedirs(os.path.dirname(INPUT_PATH), exist_ok=True)

records = [
    {"id": 1, "customer": "Asha",  "amount": 120.5, "note": 'said "urgent" twice'},
    {"id": 2, "customer": "Ben",   "amount": 0.0,   "note": "line one\nline two"},
    {"id": 3, "customer": "Chen",  "amount": 45.0,  "note": "qty\t2\tbox"},
    {"id": 4, "customer": "Diego", "amount": 300.0, "note": r"path C:\reports\aug.csv"},
    {"id": 5, "customer": "Esha",  "amount": 75.25, "note": "left, then right, then gone"},
    {"id": 6, "customer": "Farid", "amount": None,  "note": None},
    {"id": 7, "customer": "Gita",  "amount": 19.99, "note": "café — closed ☕"},
    {"amount": 210.0, "note": "keys out of order", "customer": "Hana", "id": 8},
    {"id": 9, "customer": "Iris",  "amount": 60.0},                  # no note key at all
    {} #All fields null
]

# The same text as record 2 — but never passed through json.dumps, so the newline
# stays a REAL line break and splits this record in two.
BROKEN = '{"id": 10, "customer": "Jonas", "amount": 88.0, "note": "line one\nline two"}'

# New valid record to be added after the broken one
NEW_VALID_RECORD_DICT = {"id": 11, "customer": "Wendy", "amount": 100.0, "note": "This is a new valid record"}
NEW_VALID_RECORD_STR = json.dumps(NEW_VALID_RECORD_DICT, ensure_ascii=False)


with open(INPUT_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")   # ensure_ascii=False keeps café as café
    f.write(BROKEN + "\n")
    f.write(NEW_VALID_RECORD_STR + "\n") # Write the new valid record

print("wrote", INPUT_PATH)

In [ ]:
!cat  /content/ravi-writes/data/input/orders.ndjson
!echo "---"
!wc -l /content/ravi-writes/data/input/orders.ndjson

### Records 2 and 10 hold the **same text** — one is fine, one is broken

Both notes are `line one`, a newline, `line two`.

* Record 2 went through `json.dumps`, so the newline was **escaped** into the two
  characters `\` and `n`. The record is still **one line**. This is the whole trick of
  NDJSON: escaping is what keeps a value from breaking the record boundary.
* Record 10 was hand-written with a **real** newline, so it is **two lines** — and
  neither half is valid JSON. `{"id": 10, … "note": "line one` never closes its string;
  `line two"}` starts nowhere.

That is why `wc -l` says **11** when you meant to write **10** records. The count
mismatch is the tell.

The other rows plant the rest of the mess a real feed hands you:

| id | what's in it |
| -- | ------------ |
| 1 | `"` **quotes inside** a value → escaped as `\"` |
| 3 | **tabs** → escaped as `\t` |
| 4 | **backslashes** (a Windows path) → each one doubled, `\\` |
| 5 | **commas** inside a value — free in JSON; the classic thing that ruins a CSV |
| 6 | **nulls** in `amount` *and* `note` |
| 7 | **non-ASCII** — `café — closed ☕`, real UTF-8 |
| 8 | **keys in a different order** — JSON matches by *name*, not position (unlike a DML record format) |
| 9 | **no `note` key at all** — a missing key is not an error |

**Requirement:** read `orders.ndjson` into a DataFrame with a schema **you declare** —
`id` int, `customer` string, `amount` double, `note` string — and:

1. Show that the escaped values come back as the **real characters** (`"`, tab,
   newline, `\`, `café ☕`) — not as `\"` and `\t`.
2. Get hold of the **two corrupt lines** record 10 produced, with their raw text, and
   count good rows vs bad ones. You should end up with **9 good + 2 corrupt = 11 rows**.
3. Read the same file again so that it **fails loudly** on the bad record instead of
   quietly returning nulls.

## The Ab Initio solution

_(to be written)_

## The Spark solution — step by step

Your turn to write it. The two cells below are just setup — the solution goes in the
empty cell after them.

In [ ]:
# 1. Install PySpark
!pip install -q pyspark

In [ ]:
# 2. Imports & SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType,
)

spark = SparkSession.builder.appName("read-ndjson-demo").getOrCreate()
spark

In [ ]:
INPUT_PATH = "/content/ravi-writes/data/input/orders.ndjson"

schema = StructType([
    StructField("id", IntegerType(), True), # Changed to nullable=True
    StructField("customer", StringType(), True), # Changed to nullable=True
    StructField("amount", DoubleType(), True), # Changed to nullable=True
    StructField("note", StringType(), True),
    StructField("_corrupt_record", StringType(), True) # Added this line for corrupt records
])

df_json = spark.read.format("json") \
                .schema(schema) \
                .option("mode", "PERMISSIVE") \
                .option("columnNameOfCorruptRecord", "_corrupt_record") \
                .load(INPUT_PATH)

df_json.show()

df_json.printSchema()

### Alternative: Robust Line-by-Line Parsing for NDJSON

To ensure no valid records are silently skipped after a severe corruption, we can read the file as plain text first and then explicitly try to parse each line as JSON. This gives us finer control over how malformed lines are handled.

In [ ]:
from pyspark.sql.functions import col, from_json, lit, when

# Define the schema for the *valid* JSON records (without the _corrupt_record field initially)
# We'll add _corrupt_record manually.
schema_for_parsing = StructType([
    StructField("id", IntegerType(), True),
    StructField("customer", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("note", StringType(), True),
])

# Read the file as plain text, one line per row
df_raw_lines = spark.read.text(INPUT_PATH)

# Define options as a dictionary
options = {
    "mode": "PERMISSIVE",
    "columnNameOfCorruptRecord": "_corrupt_record",
    "dropFieldIfAllNull": "true"
}

# Attempt to parse each line as JSON using from_json
df_parsed_lines = df_raw_lines.withColumn(
    "parsed_json", from_json(col("value"), schema_for_parsing, options)
)

# Now, construct the final DataFrame, distinguishing between valid and corrupt records
df_json_robust = df_parsed_lines.select(
    col("parsed_json.id").alias("id"),
    col("parsed_json.customer").alias("customer"),
    col("parsed_json.amount").alias("amount"),
    col("parsed_json.note").alias("note"),
    when((col("parsed_json.id").isNull() & \
          col("parsed_json.customer").isNull() & \
          col("parsed_json.amount").isNull() & \
          col("parsed_json.note").isNull()), col("value")).alias("_corrupt_record")
)

print("DataFrame with robust parsing:")
df_json_robust.show(truncate=False)

df_json_robust.printSchema()

### Differentiating between Malformed JSON and Valid JSON with Nulls

To strictly identify only syntactically invalid JSON as 'corrupt', we can use a UDF that attempts to parse each raw line and flags it based on parsing success or failure.

In the `df_json_robust_strict` DataFrame, only the truly malformed records will appear in `_corrupt_record`. A record like `{"id": null, "customer": null, "amount": null, "note": null}` would be considered a valid record (though with all null fields) and would not have its raw string copied to `_corrupt_record`.

You can see that with this robust parsing method, the record with `id: 11` (Wendy) is now correctly included, and the two parts of the corrupt record 10 are each captured as distinct `_corrupt_record` entries, alongside the other valid records.

In [ ]:
# Drop corrupt records

    #when((col("parsed_json.id").isNull() & \
    #      col("parsed_json.customer").isNull() & \
    #      col("parsed_json.amount").isNull() & \
    #      col("parsed_json.note").isNull()), col("value")).alias("_corrupt_record")

df_filtered_json = df_json.filter(col("id").isNotNull() & col("customer").isNotNull() & col("amount").isNotNull())
df_filtered_json.show()

In [ ]:
df_json.cache() # Cache the DataFrame to allow querying _corrupt_record

df_corrupt_records = df_json.filter(col("_corrupt_record").isNotNull())
corrupt_count = df_corrupt_records.count()

reject_threshold = 10 # Define your threshold for acceptable rejects

print(f"Total corrupt records found: {corrupt_count}")

if corrupt_count > reject_threshold:
    print(f"Warning: The number of corrupt records ({corrupt_count}) exceeds the threshold ({reject_threshold}). A quality-first pipeline would typically fail here.")
    # In a production pipeline, you might raise an exception here to halt execution:
    # raise ValueError(f"Corrupt record count ({corrupt_count}) exceeds threshold ({reject_threshold})")
else:
    print(f"Corrupt record count ({corrupt_count}) is within the acceptable threshold ({reject_threshold}).")

df_corrupt_records.select("_corrupt_record").show(truncate=False)

## Your turn

Variations to try yourself (no solutions provided):

1. **Take the safety net away.** Read the file with the same schema but *without* the
   corrupt-record column in it. Where do the two bad lines go — dropped, or still
   there? What do they look like?
2. **Ask for the wrong shape.** Read this NDJSON file with `multiLine=True` — the
   option a *pretty-printed JSON array* needs. How many rows come back, and why?
3. **Fix the broken record.** Rebuild the file with all ten records going through
   `json.dumps`, re-read it, and confirm you get 10 good rows and no corrupt column.
4. **Write it back out.** Send the good rows to `/content/ravi-writes/data/output/`
   with `df.write.json(...)`, then `!cat` the `part-*` file. Are the quotes, tabs and
   newlines escaped again on the way out — and is it still one line per record?
5. **Let Spark guess.** Drop the schema entirely and compare `printSchema()` with your
   declared one. What type did it pick for `amount`, and what did it do with record 9's
   missing `note`?